# MGT806 Week 1 — Hello, model

**AI in Business · PhD · Jinnah University for Women**

By the end of this notebook you will have called a language model from code, watched it fail,
counted what it cost, and saved a record of what you did.

**The two rules of this course**

1. You never write code from scratch. You specify what you want and an assistant writes it.
2. You never run code you have not read. If you cannot say what a cell does, do not run it.

Read every cell before you run it. If a line is unclear, that is the question to ask in class.

## 1. Install the client library

`google-genai` is the official Python package for calling Gemini models.
The `-q` flag just means 'quiet' — print less. You only need to run this once per session;
a notebook session forgets everything when it disconnects.

In [ ]:
!pip install -q google-genai

## 2. Your API key

Get a free key at **https://aistudio.google.com/apikey** — a Google account is enough, no payment card.

We use `getpass` so the key is typed into a hidden box rather than written into the notebook.
**Never paste a key directly into a cell.** A notebook is a document: you will share it, commit it to
GitHub, or email it, and the key goes with it. Treat a leaked key like a leaked password.

> **Use your own API key.** Every scholar must create and paste their **own** Gemini API key (from https://aistudio.google.com/apikey). Do not share keys or use a classmate's: the free tier's limits are per key, so a shared key runs out for everyone, and you are responsible for whatever is sent with your key.

In [ ]:
from getpass import getpass
from google import genai

API_KEY = getpass('Paste your Gemini API key (it will not be displayed): ')
client = genai.Client(api_key=API_KEY)
print('Client ready.')

### Which models can your key actually reach?

Model names change every few months and free-tier availability changes with them. Rather than trust
a name written in a notebook, ask. If a later cell fails with a model-not-found error, come back here
and pick a name from this list that contains the word `flash`.

In [ ]:
try:
    names = [m.name for m in client.models.list()]
    for n in names:
        if 'flash' in n or 'embedding' in n:
            print(n)
except Exception as e:
    print('Could not list models:', e)

## 3. Your first call

Three things are happening: we name a **model**, we send **contents** (the prompt), and we read
`.text` off the response. That is the whole shape of every call you will make this semester.

`MODEL` is a variable so you can change it in one place later.

In [ ]:
import time
from google.genai import errors

MODEL = 'gemini-3.8-flash'      # primary model; free tier as of Oct 2026: https://ai.google.dev/gemini-api/docs/pricing
FALLBACK_MODEL = 'gemini-3.5-flash'   # used automatically if the primary is overloaded (503) or rate-limited (429)

def ask(prompt, model=None, retries=3):
    """Send a prompt; on 429/503 wait briefly, then fall back to FALLBACK_MODEL."""
    models = [model] if model else [MODEL, FALLBACK_MODEL]
    last = None
    for m in models:
        for attempt in range(retries):
            try:
                return client.models.generate_content(model=m, contents=prompt)
            except (errors.ClientError, errors.ServerError) as e:
                if getattr(e, 'code', None) not in (429, 503):
                    raise
                last = e
                print(f'{m}: error {e.code}, ' + ('retrying...' if attempt < retries - 1 else 'giving up on this model.'))
                time.sleep(5 * (attempt + 1))
        if len(models) > 1 and m == MODEL:
            print(f'Switching to fallback model {FALLBACK_MODEL}.')
    raise last

response = ask('In three sentences, explain what a supply chain bullwhip effect is to an MBA student.')
print(response.text)

## 4. Change one word

Run the next cell, then change `MBA student` to `nine-year-old` and run it again.

Nothing about the model changed. The only thing that changed is the context you gave it.
This is the entire subject of prompt and context engineering, and you have just done it once.

In [ ]:
audience = 'MBA student'   # <-- change this, then run the cell again

prompt = f'In three sentences, explain what a supply chain bullwhip effect is to a {audience}.'
print(ask(prompt).text)

## 5. Make it fail on purpose

Ask for something the model cannot know: a very recent, very local fact.

Watch the *tone* of the answer, not just the content. Note whether it hedges, and whether its
confidence tracks whether it is right. Write down what you observe — this is your first
piece of evidence for Week 3, where we explain why this happens.

In [ ]:
prompt = ('What was the closing value of the KSE-100 index yesterday, '
          'and what was the single largest contributor to the move?')
print(ask(prompt).text)

### Now ask for something checkable

Ask it for three journal articles on a topic in your own field, with authors, journal and year.
Then look for each one. Keep a note of how many exist exactly as described.

Do not skip this. It is the experience the rest of the course is built on.

In [ ]:
prompt = ('List three peer-reviewed journal articles on supplier selection using fuzzy DEMATEL. '
          'For each give authors, year, article title and journal name.')
print(ask(prompt).text)

## 6. What did that cost?

Models do not read words. They read **tokens** — roughly four characters of English each.
Tokens are simultaneously how the model sees text and how you are billed. Week 2 is about the first;
this cell is about the second.

Try the same sentence in Urdu and compare the count. Hold on to that difference.

In [ ]:
english = 'Explain the bullwhip effect in supply chains.'
urdu    = 'سپلائی چین میں بل وہپ ایفیکٹ کی وضاحت کریں۔'

for label, text in [('English', english), ('Urdu', urdu)]:
    n = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f'{label:8s} {n:4d} tokens  |  {len(text):4d} characters')

## 7. Save a record of this session

Every piece of work in this course carries a workflow log: what you ran, when, with which model.
This is a course requirement, an HEC requirement, and increasingly a journal requirement.
Start the habit now, in week one, while it is trivial.

In [ ]:
from datetime import datetime, timezone

log = f'''MGT806 Week 1 workflow log
scholar: YOUR NAME HERE
date_utc: {datetime.now(timezone.utc).isoformat(timespec='seconds')}
model: {MODEL}
what I did: first API calls; tested a local factual question and a citation request
what I observed: WRITE ONE HONEST SENTENCE HERE
'''

with open('week1_log.txt', 'w', encoding='utf-8') as f:
    f.write(log)

print(log)

## 8. Put it on GitHub

In Colab: **File → Save a copy in GitHub**. Choose your `mgt806` repository, keep the filename,
and write a commit message that says what you actually did — not `update`.

A commit message is a sentence to your future self and to your examiner. `Week 1: first API calls,
citation check found 2 of 3 references fabricated` is a good one.

---

### Lab 1 — what to submit

1. This notebook, run, committed to your repository.
2. `week1_log.txt` with the observation line completed honestly.
3. In the final markdown cell below: **how many of the three citations existed exactly as described?**
   Name which were wrong and in what way — wrong author, wrong year, wrong journal, or wholly invented.

That last answer is the whole of week one. Everything else was setup.

### Your answer

_Replace this text. Three or four sentences is enough._